# 00 — Setup y Verificación del Entorno
## MEF Data Lake

Este notebook verifica que:
- La estructura de directorios esté creada.
- PySpark pueda iniciarse correctamente.
- Los datos Bronze existan antes de ejecutar los pipelines Silver/Gold.


In [ ]:
import sys
from pathlib import Path

# Agregar la raíz del proyecto al path de Python
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root : {PROJECT_ROOT}")
print(f"Python path  : OK")

In [ ]:
from src.paths import ensure_dirs, BRONZE, SILVER, GOLD, STATIC, DATA_ROOT
from src.spark_utils import get_spark

# Crear estructura de directorios si no existe
ensure_dirs()
print(f"\n📁 Directorio de datos: {DATA_ROOT}")

## 1. Verificación de Datos Bronze

La capa Bronze se genera ejecutando desde la raíz del proyecto:
```bash
00_bronze_ingestion.ipynb
```

In [ ]:
import os

print("\n📦 Estado de la Capa Bronze:")
print("-" * 50)

for source, path in BRONZE.items():
    if path.exists():
        json_files = list(path.rglob("*.json"))
        size_mb = sum(f.stat().st_size for f in json_files) / 1_048_576
        print(f"  ✅ {source:<12} | {len(json_files):>4} archivos JSON | {size_mb:>8.1f} MB")
    else:
        print(f"  ❌ {source:<12} | directorio vacío — ejecutar 00_bronze_ingestion.ipynb")

print("\n📂 Estado de la Capa Silver:")
print("-" * 50)
for source, path in SILVER.items():
    if path.exists() and any(path.rglob("*.parquet")):
        parquet_files = list(path.rglob("*.parquet"))
        size_mb = sum(f.stat().st_size for f in parquet_files) / 1_048_576
        print(f"  ✅ {source:<12} | {len(parquet_files):>4} archivos Parquet | {size_mb:>8.1f} MB")
    else:
        print(f"  ⏳ {source:<12} | pendiente — ejecutar notebook 01/02/03")

print("\n🏅 Estado de la Capa Gold:")
print("-" * 50)
gold_ready = 0
for name, path in GOLD.items():
    if path.exists() and any(path.rglob("*.parquet")):
        gold_ready += 1
        print(f"  ✅ {name}")
    else:
        print(f"  ⏳ {name}")
print(f"\n  {gold_ready}/{len(GOLD)} tablas Gold listas")

## 2. Test de SparkSession

In [ ]:
spark = get_spark(app_name="MEF_Setup_Check", memory="2g")

print(f"\n🔥 Spark Info:")
print(f"  Versión  : {spark.version}")
print(f"  Master   : {spark.sparkContext.master}")
print(f"  App Name : {spark.sparkContext.appName}")

# Test rápido: crear un DataFrame simple
test_df = spark.range(5)
print(f"\n✅ Spark funcional — {test_df.count()} registros de prueba")

spark.stop()
print("✅ SparkSession detenida correctamente")

## 3. Orden de Ejecución del Pipeline

```
BRONZE (extracción)
  └─► 00_bronze_ingestion.ipynb

SILVER (transformación)
  ├─► 01_silver_siaf.ipynb
  ├─► 02_silver_sismepre.ipynb
  └─► 03_silver_renamu.ipynb

GOLD (modelado dimensional)
  ├─► 04_gold_siaf.ipynb
  ├─► 05_gold_sismepre.ipynb
  └─► 06_gold_renamu.ipynb

ANÁLISIS
  └─► 07_analisis_exploratorio.ipynb
```

> 💡 Cada notebook es independiente y autocontenido. Puede ejecutarse por separado.
